In [59]:
"""
Compara las agrupaciones de sentido del LLM juez (llm_juez.py) contra los
clusters de HDBSCAN de cada modelo (clusterizar_sentidos.py), usando Adjusted
Rand Index (ARI) y Normalized Mutual Information (NMI) -- métricas estándar
para comparar dos particiones del MISMO conjunto de elementos sin que importe
el nombre/orden de las etiquetas (a diferencia de Cohen's Kappa, que necesita
etiquetas ya alineadas).
 
Esta es la validación externa que quedó pendiente desde el inicio del
protocolo (sección 8 original): antes no había un ground truth confiable a
escala para calcularla; ahora el LLM juez lo provee.
 
Requiere haber corrido antes: llm_juez.py (juicios_llm/<palabra>.json) y
tener embeddings_dict/tweets_dict construidos (ver construir_dicts_para_palabras
en clusterizar_sentidos.py) para las mismas palabras, en los 5 modelos.
 
Uso:
    resultados = comparar_lote(["entrada", "elefante", "torres", "pagar"],
                                modelos=["mex_state","mex_large","beto","bertin","robertuito"])
"""
 
import json
import os
 
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import umap
import hdbscan
import unicodedata
from sklearn.metrics import silhouette_score
from collections import defaultdict
import glob

In [70]:
CARPETA_JUICIOS = "juicios_llm_qwen_model_"
CARPETA_EMBEDDINGS = "embeddings"

UMAP_N_COMPONENTS = 12
HDBSCAN_MIN_SAMPLES = 3
MIN_OCURRENCIAS_PARA_CLUSTER = 8   # palabras con menos ocurrencias no se agrupan

MIN_TWEETS_POR_SENTIDO_LLM = 5
 
ETIQUETA_RUIDO_LLM = "ruido_llm"

In [78]:
MCS_FRACCION = 0.05   # min_cluster_size ≈ 5% de las ocurrencias de la palabra
MCS_MINIMO = 5
MCS_MAXIMO = 50

def calcular_min_cluster_size(n, fraccion=MCS_FRACCION, minimo=MCS_MINIMO, maximo=MCS_MAXIMO):
    return int(np.clip(round(fraccion * n), minimo, maximo))
    
def cargar_embeddings(nombre_modelo, carpeta=CARPETA_EMBEDDINGS):
    ruta = os.path.join(carpeta, f"{nombre_modelo}_embeddings.npz")
    data = np.load(ruta, allow_pickle=True)
    emb = data["embeddings"]
    words = data["words"]
    id_tweets = data["id_tweets"]
    return emb, words, id_tweets
    
def cargar_juicio_llm(palabra, carpeta=CARPETA_JUICIOS):
    ruta = os.path.join(carpeta, f"{palabra}.json")
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)
 
def inspeccionar_ocurrencias(palabra, embeddings_dict, tweets_dict, min_cluster_size=None, min_samples=None,
                               n_componentes_quitar=0, semilla=42):
    """Devuelve un DataFrame (id_tweet, cluster_id) para revisar caso por caso
    a qué tweet corresponde cada ocurrencia y en qué cluster cayó (cluster,
    no "sentido" — ver limitación en el encabezado del notebook). Usa UNA sola
    semilla (para tener labels concretos que leer) -- si quieres confirmar que
    el k de esa semilla es representativo (la moda), corre antes
    medir_estabilidad.py para esa palabra."""
    embs = embeddings_dict[palabra]
    tids = tweets_dict[palabra]
    mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(len(embs))
    ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
    labels, _ = _hdbscan_labels(embs, mcs, ms, n_componentes_quitar=n_componentes_quitar, semilla=semilla)
    return pd.DataFrame({"id_tweet": tids, "cluster_id": labels})

    
def comparar_palabra_modelo(palabra, nombre_modelo, embeddings_dict, tweets_dict,
                              inspeccionar_ocurrencias_fn, carpeta_juicios=CARPETA_JUICIOS,
                              min_tweets_sentido_llm=MIN_TWEETS_POR_SENTIDO_LLM, semilla=42):
    """Compara, para UNA palabra y UN modelo, la partición del LLM juez contra
    la de HDBSCAN. `inspeccionar_ocurrencias_fn` es la función del mismo
    nombre definida en clusterizar_sentidos.py (se pasa como parámetro para
    no duplicar código ni imports pesados de umap/hdbscan en este archivo).
 
    Los sentidos del LLM con menos de `min_tweets_sentido_llm` tweets
    asignados (en toda la muestra juzgada, no solo en el cruce con el modelo)
    se tratan como ruido -- igual que el -1 de HDBSCAN -- y se excluyen de la
    comparación."""
    juicio = cargar_juicio_llm(palabra, carpeta_juicios)
    n_sentidos_llm_total = len(juicio.get("sentidos", []))
    asignaciones_llm, sentidos_ruido, n_sentidos_llm_5plus = _asignaciones_con_ruido_llm(
        juicio, min_tweets=min_tweets_sentido_llm
    )
 
    tabla_modelo = inspeccionar_ocurrencias_fn(palabra, embeddings_dict, tweets_dict, semilla=semilla)  # id_tweet, cluster_id
 
    # comparar solo sobre tweets presentes en AMBOS: la muestra que vio el LLM
    # y las ocurrencias que efectivamente tiene el modelo
    tabla_modelo = tabla_modelo[tabla_modelo["id_tweet"].isin(asignaciones_llm.keys())].copy()
    tabla_modelo["sentido_llm"] = tabla_modelo["id_tweet"].map(asignaciones_llm)
 
    # excluir "no_claro" y "ruido_llm" del LLM, y ruido (-1) del modelo --
    # ninguno de los tres es una etiqueta de sentido/cluster real; incluirlos
    # infla artificialmente el acuerdo o el desacuerdo según el caso
    comparable = tabla_modelo[
        (tabla_modelo["sentido_llm"] != "no_claro")
        & (tabla_modelo["sentido_llm"] != ETIQUETA_RUIDO_LLM)
        & (tabla_modelo["cluster_id"] != -1)
    ]
    n_excluidos_ruido_llm = int(
        (tabla_modelo["sentido_llm"] == ETIQUETA_RUIDO_LLM).sum()
    )
 
    base = {
        "palabra": palabra, "modelo": nombre_modelo,
        "n_sentidos_llm_total": n_sentidos_llm_total,
        "n_sentidos_llm_5plus": n_sentidos_llm_5plus,
        "n_sentidos_llm_ruido": len(sentidos_ruido),
        "n_excluidos_ruido_llm": n_excluidos_ruido_llm,
    }
 
    if len(comparable) < 5 or comparable["sentido_llm"].nunique() < 2:
        # no hay suficiente base, o el LLM no encontró más de un sentido real
        # (tras descontar ruido) -- ARI/NMI no son informativos con una sola clase
        return {
            **base,
            "n_comparado": len(comparable),
            "n_clusters_modelo": tabla_modelo["cluster_id"].nunique(),
            "ari": np.nan, "nmi": np.nan,
        }
 
    labels_llm = comparable["sentido_llm"].tolist()
    labels_modelo = comparable["cluster_id"].tolist()
 
    ari = adjusted_rand_score(labels_llm, labels_modelo)
    nmi = normalized_mutual_info_score(labels_llm, labels_modelo)
    n_clusters_modelo = len(set(labels_modelo))
 
    return {
        **base,
        "n_comparado": len(comparable),
        "n_clusters_modelo": n_clusters_modelo, "ari": ari, "nmi": nmi,
    }
    
def _hdbscan_labels(embeddings, min_cluster_size, min_samples, n_componentes_quitar=0, semilla=42):
    """Corre UMAP+HDBSCAN sobre los embeddings de una palabra y devuelve
    (labels, silhouette). labels: array de labels (uno por ocurrencia, -1 =
    ruido). silhouette: score de separación entre clusters (None si k<2 o muy
    pocos puntos no-ruido). Si n_componentes_quitar>0, primero centra y quita
    esa(s) componente(s) principal(es) dominante(s) (ver
    quitar_componentes_dominantes)."""
    X = np.vstack(embeddings)
    if n_componentes_quitar > 0:
        X = quitar_componentes_dominantes(X, n_componentes_quitar)
    n = len(embeddings)
    n_components = min(UMAP_N_COMPONENTS, n - 2)
    n_neighbors = min(15, n - 2)
    # n_jobs=1 es NECESARIO además de random_state -- UMAP paraleliza partes del
    # cálculo (búsqueda de vecinos, optimización) por defecto, y eso puede dar
    # resultados distintos entre corridas incluso con semilla fija. Sin esto,
    # el número de clusters puede variar (se detectó variación de 2 a 8 para la
    # misma palabra/modelo entre corridas) -- invalida cualquier lectura manual
    # de resultados si no se fuerza un solo hilo.
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors, random_state=semilla, n_jobs=1)
    X_red = reducer.fit_transform(X)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples)
    labels = clusterer.fit_predict(X_red)
 
    k = len(set(labels) - {-1})
    mask = labels != -1
    if k >= 2 and mask.sum() >= k + 1:
        silhouette = silhouette_score(X_red[mask], labels[mask])
    else:
        silhouette = None
    return labels, silhouette
    
def _asignaciones_con_ruido_llm(juicio, min_tweets=MIN_TWEETS_POR_SENTIDO_LLM):
    """Devuelve (asignaciones_llm, sentidos_ruido, n_sentidos_5plus).
 
    asignaciones_llm: dict id_tweet -> sentido, con los sentidos que tienen
    menos de `min_tweets` tweets asignados (contando TODA la muestra que vio
    el juez, no solo la que se cruza con el modelo) re-etiquetados como
    ETIQUETA_RUIDO_LLM.
    sentidos_ruido: el conjunto de ids de sentido tratados como ruido.
    n_sentidos_5plus: cuántos sentidos declarados por el LLM tienen >=
    min_tweets tweets -- el conteo "real" de sentidos una vez descontado el
    ruido, más comparable con el criterio que ya se usa para decidir si un
    modelo de clustering "presenta múltiples clusters" (sección 8/9).
    """
    conteo = {}
    for a in juicio["asignaciones"]:
        conteo[a["sentido"]] = conteo.get(a["sentido"], 0) + 1
 
    sentidos_ruido = {
        sid for sid, n in conteo.items()
        if sid != "no_claro" and n < min_tweets
    }
    n_sentidos_5plus = sum(
        1 for sid in (s["id"] for s in juicio.get("sentidos", []))
        if conteo.get(sid, 0) >= min_tweets
    )
 
    asignaciones_llm = {
        a["id_tweet"]: (ETIQUETA_RUIDO_LLM if a["sentido"] in sentidos_ruido else a["sentido"])
        for a in juicio["asignaciones"]
    }
    return asignaciones_llm, sentidos_ruido, n_sentidos_5plus
 
 
def comparar_lote(palabras, cargar_embeddings_fn, construir_dicts_fn, inspeccionar_ocurrencias_fn,
                    modelos=("mex_state", "mex_large", "beto", "bertin", "robertuito"),
                    carpeta_juicios=CARPETA_JUICIOS,
                    min_tweets_sentido_llm=MIN_TWEETS_POR_SENTIDO_LLM,
                  semilla=42
                 ):
    """Corre comparar_palabra_modelo para cada combinación de palabra x modelo.
    `construir_dicts_fn` es construir_dicts_para_palabras() de
    clusterizar_sentidos.py (se pasa como parámetro por la misma razón que
    arriba). `min_tweets_sentido_llm` es el piso de tweets para que un sentido
    del LLM cuente como real en vez de ruido (ver MIN_TWEETS_POR_SENTIDO_LLM)."""
    resultados = []
    for nombre_modelo in modelos:
        emb_dict, tw_dict = construir_dicts_fn(nombre_modelo, set(palabras))
        for palabra in palabras:
            if palabra not in emb_dict:
                print(f"  [{nombre_modelo}] '{palabra}' no está en los embeddings -- se salta")
                continue
            r = comparar_palabra_modelo(palabra, nombre_modelo, emb_dict, tw_dict,
                                          inspeccionar_ocurrencias_fn, carpeta_juicios,
                                          min_tweets_sentido_llm=min_tweets_sentido_llm, 
                                        semilla=semilla)
            resultados.append(r)
 
    df = pd.DataFrame(resultados)
    df.to_csv("comparacion_llm_vs_clustering.csv", index=False)
 
    n_ruido_total = df["n_sentidos_llm_ruido"].sum()
    n_tweets_excluidos = df["n_excluidos_ruido_llm"].sum()
    print(f"Sentidos del LLM tratados como ruido (< {min_tweets_sentido_llm} tweets): "
          f"{n_ruido_total} en total, {n_tweets_excluidos} tweets excluidos de la comparación\n")
 
    print("=== Comparación LLM juez vs. clustering, por modelo ===")
    print(df.groupby("modelo")[["ari", "nmi"]].describe([0.5]).to_string())
 
    for col_sentidos, etiqueta in [
        ("n_sentidos_llm_total", "sin descontar ruido (crudo)"),
        ("n_sentidos_llm_5plus", f"descontando ruido (sentidos con >={min_tweets_sentido_llm} tweets)"),
    ]:
        print(f"\n=== Correlación entre {col_sentidos} y n_clusters_modelo, por modelo -- {etiqueta} ===")
        for modelo, grupo in df.groupby("modelo"):
            valido = grupo.dropna(subset=[col_sentidos, "n_clusters_modelo"])
            if len(valido) >= 3:
                corr = valido[col_sentidos].corr(valido["n_clusters_modelo"], method="spearman")
                print(f"  {modelo}: rho={corr:.3f} (n={len(valido)})")
 
    return df

In [75]:
def construir_dicts(nombre_modelo, palabras):
    emb, w, id_tweets= cargar_embeddings(nombre_modelo)
    emb /= np.linalg.norm(emb, axis=1).reshape(-1, 1)

    # normaliza a NFC -- "ñ" puede venir como un solo carácter o como "n" + tilde
    # combinada; se ven idénticas pero no son el mismo string en Python
    palabras_arr = np.array([unicodedata.normalize("NFC", str(d)) for d in w])
    tweets_arr = np.array(id_tweets)

    emb_dict, tw_dict = defaultdict(list), defaultdict(list)
    for palabra in palabras:
        palabra_nfc = unicodedata.normalize("NFC", palabra)
        idx = np.where(palabras_arr == palabra_nfc)[0]
        if len(idx) == 0:
            print(f"  [{nombre_modelo}] '{palabra}' no está en el .npz guardado — sáltala o revisa el nombre")
            continue
        emb_dict[palabra] = list(emb[idx])
        tw_dict[palabra] = list(tweets_arr[idx])
    #embeddings_calib_por_modelo[nombre_modelo] = emb_dict
    #tweets_calib_por_modelo[nombre_modelo] = tw_dict
    return emb_dict, tw_dict

In [76]:
files = glob.glob("juicios_llm_qwen_model_/*.json")
palabras = [f.split("\\")[-1][0:-5] for f in files]
MODELOS = ["mex_state", "mex_large", "beto", "bertin", "robertuito"]
cargar_embeddings_fn = cargar_embeddings
inspeccionar_ocurrencias_fn = inspeccionar_ocurrencias
construir_dicts_fn = construir_dicts

In [84]:


df = comparar_lote(palabras, cargar_embeddings_fn, construir_dicts_fn, inspeccionar_ocurrencias_fn,
                    modelos=MODELOS,
                    carpeta_juicios=CARPETA_JUICIOS, semilla=0)

Sentidos del LLM tratados como ruido (< 5 tweets): 470 en total, 863 tweets excluidos de la comparación

=== Comparación LLM juez vs. clustering, por modelo ===
             ari                                                     nmi                                             
           count      mean       std       min       50%       max count      mean       std  min       50%       max
modelo                                                                                                               
bertin      39.0  0.112415  0.203535 -0.162325  0.037436  0.698675  39.0  0.180397  0.181208  0.0  0.094896  0.696286
beto        39.0  0.211583  0.267711 -0.111270  0.143106  1.000000  39.0  0.322493  0.261940  0.0  0.269153  1.000000
mex_large   39.0  0.211547  0.279032 -0.080061  0.103112  1.000000  39.0  0.277797  0.253707  0.0  0.180370  1.000000
mex_state   39.0  0.162231  0.214771 -0.083654  0.129441  0.851914  39.0  0.257507  0.215550  0.0  0.210604  0.763581
robertuito  3

In [79]:

df = comparar_lote(palabras, cargar_embeddings_fn, construir_dicts_fn, inspeccionar_ocurrencias_fn,
                    modelos=MODELOS,
                    carpeta_juicios=CARPETA_JUICIOS, semilla=1)

Sentidos del LLM tratados como ruido (< 5 tweets): 470 en total, 863 tweets excluidos de la comparación

=== Comparación LLM juez vs. clustering, por modelo ===
             ari                                                     nmi                                             
           count      mean       std       min       50%       max count      mean       std  min       50%       max
modelo                                                                                                               
bertin      39.0  0.129757  0.215608 -0.115532  0.055187  0.698675  39.0  0.188639  0.193939  0.0  0.113707  0.696286
beto        39.0  0.233933  0.278858 -0.116824  0.180453  1.000000  39.0  0.326489  0.274584  0.0  0.297375  1.000000
mex_large   39.0  0.219390  0.286827 -0.080061  0.103112  1.000000  39.0  0.282310  0.257884  0.0  0.182885  1.000000
mex_state   39.0  0.169492  0.233372 -0.118101  0.121230  0.851914  39.0  0.259680  0.224007  0.0  0.185276  0.763581
robertuito  3

In [80]:

df = comparar_lote(palabras, cargar_embeddings_fn, construir_dicts_fn, inspeccionar_ocurrencias_fn,
                    modelos=MODELOS,
                    carpeta_juicios=CARPETA_JUICIOS, semilla=2)

Sentidos del LLM tratados como ruido (< 5 tweets): 470 en total, 863 tweets excluidos de la comparación

=== Comparación LLM juez vs. clustering, por modelo ===
             ari                                                     nmi                                             
           count      mean       std       min       50%       max count      mean       std  min       50%       max
modelo                                                                                                               
bertin      39.0  0.122861  0.208885 -0.104513  0.030441  0.698675  39.0  0.188072  0.195462  0.0  0.094896  0.696286
beto        39.0  0.237079  0.279360 -0.111270  0.147195  1.000000  39.0  0.330736  0.256250  0.0  0.318086  1.000000
mex_large   39.0  0.218113  0.288457 -0.068008  0.116886  1.000000  39.0  0.281187  0.263227  0.0  0.175234  1.000000
mex_state   39.0  0.170653  0.239431 -0.088028  0.097767  0.851914  39.0  0.258359  0.223199  0.0  0.185276  0.763581
robertuito  3

In [81]:

df = comparar_lote(palabras, cargar_embeddings_fn, construir_dicts_fn, inspeccionar_ocurrencias_fn,
                    modelos=MODELOS,
                    carpeta_juicios=CARPETA_JUICIOS, semilla=3)

Sentidos del LLM tratados como ruido (< 5 tweets): 470 en total, 863 tweets excluidos de la comparación

=== Comparación LLM juez vs. clustering, por modelo ===
             ari                                                     nmi                                             
           count      mean       std       min       50%       max count      mean       std  min       50%       max
modelo                                                                                                               
bertin      39.0  0.126019  0.206630 -0.104513  0.055187  0.698675  39.0  0.198057  0.207588  0.0  0.094896  0.696286
beto        39.0  0.223936  0.284994 -0.110891  0.158523  1.000000  39.0  0.314427  0.265812  0.0  0.280514  1.000000
mex_large   39.0  0.188259  0.252082 -0.085797  0.103112  0.817680  39.0  0.265092  0.234318  0.0  0.182796  0.762676
mex_state   39.0  0.165499  0.243418 -0.118101  0.097716  0.851914  39.0  0.249418  0.224035  0.0  0.185276  0.763581
robertuito  3

In [82]:

df = comparar_lote(palabras, cargar_embeddings_fn, construir_dicts_fn, inspeccionar_ocurrencias_fn,
                    modelos=MODELOS,
                    carpeta_juicios=CARPETA_JUICIOS, semilla=42)

Sentidos del LLM tratados como ruido (< 5 tweets): 470 en total, 863 tweets excluidos de la comparación

=== Comparación LLM juez vs. clustering, por modelo ===
             ari                                                     nmi                                             
           count      mean       std       min       50%       max count      mean       std  min       50%       max
modelo                                                                                                               
bertin      39.0  0.129538  0.203734 -0.104513  0.055187  0.698675  39.0  0.197113  0.193152  0.0  0.119481  0.696286
beto        39.0  0.221906  0.271545 -0.107383  0.182080  1.000000  39.0  0.311893  0.254484  0.0  0.295832  1.000000
mex_large   39.0  0.210330  0.283480 -0.095763  0.103112  1.000000  39.0  0.278069  0.259338  0.0  0.173340  1.000000
mex_state   39.0  0.162128  0.242062 -0.098025  0.088285  0.851914  39.0  0.244852  0.225782  0.0  0.173606  0.763581
robertuito  3